## Crop & mask
This notebook is intended to crop the images detected by YOLO using the returned label coordinates, and also to create the corresponding masks.

In [4]:
from ultralytics import YOLO
import csv
import cv2
import glob
import os
import numpy as np

In [2]:
# -------- CONFIG --------
model_path = './../models/YOLOv8_best.pt'
image_folder = './../data/processed/dataset/images/train/*.png'
save_root = "./../data/external/label_cutouts/"

os.makedirs(save_root, exist_ok=True)

### Crop

In [3]:
targets = {
    "red blood cell": 40,
    "leukocyte": 20,
    "trophozoite": 15,
    "schizont": 15,
    "ring": 15,
    "gametocyte": 15
}

counts = {cls: 0 for cls in targets}

model = YOLO(model_path)
image_list = glob.glob(image_folder)

print(f"Total images found: {len(image_list)}")

for img_path in image_list:

    image = cv2.imread(img_path)
    results = model.predict(image, conf=0.30, verbose=False)

    for result in results:
        boxes = result.boxes

        for box, cls_id in zip(boxes.xyxy, boxes.cls):

            cls_id = int(cls_id.cpu().numpy())
            class_name = model.names.get(cls_id)

            if class_name not in targets:
                continue

            # only save if the target has not been reached yet
            if counts[class_name] < targets[class_name]:

                x1, y1, x2, y2 = map(int, box.cpu().numpy())
                crop = image[y1:y2, x1:x2]

                if crop.size == 0:
                    continue

                class_dir = os.path.join(save_root, class_name)
                os.makedirs(class_dir, exist_ok=True)

                save_path = os.path.join(
                    class_dir,
                    f"{class_name}_{counts[class_name]}.jpg"
                )

                cv2.imwrite(save_path, crop)
                counts[class_name] += 1

print("\nCollection finished!\n")
for cls in counts:
    print(f"{cls}: {counts[cls]} images found (target: {targets[cls]})")


Total images found: 1208

Collection finished!

red blood cell: 40 images found (target: 40)
leukocyte: 20 images found (target: 20)
trophozoite: 15 images found (target: 15)
schizont: 5 images found (target: 15)
ring: 15 images found (target: 15)
gametocyte: 10 images found (target: 15)


### Mask

In [4]:
if False:
    
    input_root = "./../data/external/label_cutouts/"
    mask_root = "./../data/external/output_masks"
    seg_root = "./../data/external/output_segmented"

    image_paths = glob.glob(
        os.path.join(input_root, "**", "*.jpg"),
        recursive=True
    )

    print(f"Total images found: {len(image_paths)}")

    for img_path in image_paths:

        img = cv2.imread(img_path)
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        blur = cv2.GaussianBlur(gray, (5,5), 0)

        _, thresh = cv2.threshold(
            blur, 0, 255,
            cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )

        kernel = np.ones((3,3), np.uint8)
        opening = cv2.morphologyEx(
            thresh, cv2.MORPH_OPEN,
            kernel, iterations=2
        )

        sure_bg = cv2.dilate(opening, kernel, iterations=3)

        dist = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
        _, sure_fg = cv2.threshold(
            dist, 0.5 * dist.max(), 255, 0
        )
        sure_fg = np.uint8(sure_fg)

        unknown = cv2.subtract(sure_bg, sure_fg)

        _, markers = cv2.connectedComponents(sure_fg)
        markers = markers + 1
        markers[unknown == 255] = 0

        markers = cv2.watershed(img, markers)

        mask = np.zeros(gray.shape, dtype=np.uint8)
        mask[markers > 1] = 255

        segmented = cv2.bitwise_and(img, img, mask=mask)

        relative_path = os.path.relpath(img_path, input_root)

        mask_save_path = os.path.join(mask_root, relative_path)
        seg_save_path = os.path.join(seg_root, relative_path)

        os.makedirs(os.path.dirname(mask_save_path), exist_ok=True)
        os.makedirs(os.path.dirname(seg_save_path), exist_ok=True)

        cv2.imwrite(mask_save_path, mask)

        cv2.imwrite(seg_save_path, segmented)

    print("Finished! Masks and segmented images saved separately.")


# NOTE: This second part of the code generates masks based on Gaussian filtering and
#       the Otsu method; however, due to lighting conditions and sensitivity to stronger
#       brightness variations, the masks were not generated correctly.

In [20]:
folder = "red blood cell"
base_path = "./../data/external/label_cutouts"
output_base = f"./../data/external/output_segmented/{folder}"

os.makedirs(output_base, exist_ok=True)

for i in range(41):
    sub_folder = f"red blood cell_{i}"

    img_path = os.path.join(base_path, folder, sub_folder, "img.png")
    mask_path = os.path.join(base_path, folder, sub_folder, "label.png")

    img = cv2.imread(img_path)
    mask = cv2.imread(mask_path, 0)

    if img is None or mask is None:
        print(f"Files not found {sub_folder}")
        continue

    mask_binary = mask > 0
    output = np.zeros_like(img)
    output[mask_binary] = img[mask_binary]

    output_path = os.path.join(output_base, f"{sub_folder}.png")

    cv2.imwrite(output_path, output)

    print(f"Save: {output_path}")


Save: ./../data/external/output_segmented/red blood cell/red blood cell_0.png
Save: ./../data/external/output_segmented/red blood cell/red blood cell_1.png
Save: ./../data/external/output_segmented/red blood cell/red blood cell_2.png
Save: ./../data/external/output_segmented/red blood cell/red blood cell_3.png
Save: ./../data/external/output_segmented/red blood cell/red blood cell_4.png
Save: ./../data/external/output_segmented/red blood cell/red blood cell_5.png
Save: ./../data/external/output_segmented/red blood cell/red blood cell_6.png
Save: ./../data/external/output_segmented/red blood cell/red blood cell_7.png
Save: ./../data/external/output_segmented/red blood cell/red blood cell_8.png
Save: ./../data/external/output_segmented/red blood cell/red blood cell_9.png
Save: ./../data/external/output_segmented/red blood cell/red blood cell_10.png
Save: ./../data/external/output_segmented/red blood cell/red blood cell_11.png
Save: ./../data/external/output_segmented/red blood cell/red b

[ WARN:0@12219.436] global loadsave.cpp:278 findDecoder imread_('./../data/external/label_cutouts/red blood cell/red blood cell_19/img.png'): can't open/read file: check file path/integrity
[ WARN:0@12219.436] global loadsave.cpp:278 findDecoder imread_('./../data/external/label_cutouts/red blood cell/red blood cell_19/label.png'): can't open/read file: check file path/integrity
[ WARN:0@12219.444] global loadsave.cpp:278 findDecoder imread_('./../data/external/label_cutouts/red blood cell/red blood cell_26/img.png'): can't open/read file: check file path/integrity
[ WARN:0@12219.444] global loadsave.cpp:278 findDecoder imread_('./../data/external/label_cutouts/red blood cell/red blood cell_26/label.png'): can't open/read file: check file path/integrity
[ WARN:0@12219.445] global loadsave.cpp:278 findDecoder imread_('./../data/external/label_cutouts/red blood cell/red blood cell_28/img.png'): can't open/read file: check file path/integrity
[ WARN:0@12219.445] global loadsave.cpp:278 fi

In [5]:
data = [
    ["gametocyte_0.png", "infected", "gametocyte", "1000x", "Giemsa"],
    ["gametocyte_1.png", "infected", "gametocyte", "1000x", "Giemsa"],
    ["gametocyte_2.png", "infected", "gametocyte", "1000x", "Giemsa"],
    ["gametocyte_3.png", "infected", "gametocyte", "1000x", "Giemsa"],
    ["gametocyte_4.png", "infected", "gametocyte", "1000x", "Giemsa"],
    ["gametocyte_5.png", "infected", "gametocyte", "1000x", "Giemsa"],
    ["gametocyte_6.png", "infected", "gametocyte", "1000x", "Giemsa"],
    ["gametocyte_7.png", "infected", "gametocyte", "1000x", "Giemsa"],
    ["gametocyte_8.png", "infected", "gametocyte", "1000x", "Giemsa"],
    ["gametocyte_9.png", "infected", "gametocyte", "1000x", "Giemsa"],
    ["ring_0.png", "infected", "ring", "1000x", "Giemsa"],
    ["ring_2.png", "infected", "ring", "1000x", "Giemsa"],
    ["ring_3.png", "infected", "ring", "1000x", "Giemsa"],
    ["ring_4.png", "infected", "ring", "1000x", "Giemsa"],
    ["ring_5.png", "infected", "ring", "1000x", "Giemsa"],
    ["ring_6.png", "infected", "ring", "1000x", "Giemsa"],
    ["ring_7.png", "infected", "ring", "1000x", "Giemsa"],
    ["ring_8.png", "infected", "ring", "1000x", "Giemsa"],
    ["ring_10.png", "infected", "ring", "1000x", "Giemsa"],
    ["ring_11.png", "infected", "ring", "1000x", "Giemsa"],
    ["ring_13.png", "infected", "ring", "1000x", "Giemsa"],
    ["ring_14.png", "infected", "ring", "1000x", "Giemsa"],
    ["schizont_0.png", "infected", "schizont", "1000x", "Giemsa"],
    ["schizont_1.png", "infected", "schizont", "1000x", "Giemsa"],
    ["schizont_2.png", "infected", "schizont", "1000x", "Giemsa"],
    ["schizont_3.png", "infected", "schizont", "1000x", "Giemsa"],
    ["schizont_4.png", "infected", "schizont", "1000x", "Giemsa"],
    ["trophozoite_0.png", "infected", "trophozoite", "1000x", "Giemsa"],
    ["trophozoite_2.png", "infected", "trophozoite", "1000x", "Giemsa"],
    ["trophozoite_3.png", "infected", "trophozoite", "1000x", "Giemsa"],
    ["trophozoite_4.png", "infected", "trophozoite", "1000x", "Giemsa"],
    ["trophozoite_5.png", "infected", "trophozoite", "1000x", "Giemsa"],
    ["trophozoite_7.png", "infected", "trophozoite", "1000x", "Giemsa"],
    ["trophozoite_9.png", "infected", "trophozoite", "1000x", "Giemsa"],
    ["trophozoite_10.png", "infected", "trophozoite", "1000x", "Giemsa"],
    ["trophozoite_11.png", "infected", "trophozoite", "1000x", "Giemsa"],
    ["trophozoite_13.png", "infected", "trophozoite", "1000x", "Giemsa"],
    ["trophozoite_14.png", "infected", "trophozoite", "1000x", "Giemsa"],
    ["red blood cell_0.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_1.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_2.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_3.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_4.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_5.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_6.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_7.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_8.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_9.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_10.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_11.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_12.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_13.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_14.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_15.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_16.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_17.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_18.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_20.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_21.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_22.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_23.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_24.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_25.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_27.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_29.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_30.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_31.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_32.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_33.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_34.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_35.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_36.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_37.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_38.png", "uninfected", "red blood cell", "1000x", "Giemsa"],
    ["red blood cell_39.png", "uninfected", "red blood cell", "1000x", "Giemsa"]
]

with open("metadata.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "label", "stage", "magnification", "stain"])
    writer.writerows(data)
